# Data Ingestion with Lakeflow Connect
## Laboratório prático: JSON, Auto Loader, Lakeflow Pipelines e Lakeflow Connect

Este notebook foi montado para praticar os tópicos de maior prioridade da seção de **Data Ingestion with Lakeflow Connect**, com foco especial em:

- dados semiestruturados e JSON;
- diferença entre `coluna:campo` e `coluna.campo`;
- `from_json`, `to_json`, `get_json_object` e `parse_json`;
- Auto Loader;
- directory listing vs file notification / managed file events;
- datasets oficiais do Databricks para treino;
- Lakeflow Declarative Pipelines;
- Lakeflow Connect, incluindo managed connectors e standard connectors;
- um laboratório opcional com Google Drive.

> **Atualização das referências:** 24/09/2026.  
> Os links no final apontam para a documentação oficial do Databricks.

### Como usar

1. Importe este `.ipynb` no Databricks.
2. Conecte o notebook a um compute compatível.
3. Execute os blocos em ordem.
4. Algumas células são apenas modelos e estão explicitamente marcadas como **não executar sem configuração prévia**.
5. Se `/databricks-datasets` não estiver disponível no seu workspace, consulte o bloco de datasets de amostra para alternativas no catálogo `samples`.

## Mapa do laboratório

| Prioridade | Dataset / recurso | Caminho | Objetivo |
|---|---|---|---|
| ⭐⭐⭐ | NYC Taxi JSON | `/databricks-datasets/nyctaxi/sample/json/` | JSON, `:`, `.`, `from_json`, `to_json`, `get_json_object`, `parse_json` |
| ⭐⭐⭐ | Structured Streaming Events | `/databricks-datasets/structured-streaming/events` | Auto Loader, checkpoint, ingestão incremental |
| ⭐⭐ | Retail Sales Orders | `/databricks-datasets/retail-org/sales_orders` | JSON + Auto Loader + Lakeflow Pipelines |
| ⭐⭐ | Retail Customers | `/databricks-datasets/retail-org/customers/` | CSV + ingestão + join + Bronze/Silver |
| ⭐⭐ | Google Drive | conexão Unity Catalog | Lakeflow Connect managed vs standard connector |

O Databricks também disponibiliza datasets no Unity Catalog em `samples`, inclusive o volume:

```text
/Volumes/samples/databricks/datasets/
```

Documentação oficial:  
https://docs.databricks.com/aws/en/discover/databricks-datasets

In [0]:
# Descoberta rápida dos datasets disponíveis no workspace.

paths_to_check = [
    "/databricks-datasets/nyctaxi/sample/json/",
    "/databricks-datasets/structured-streaming/events",
    "/databricks-datasets/retail-org/sales_orders",
    "/databricks-datasets/retail-org/customers/",
]

for path in paths_to_check:
    print(f"\n--- {path}")
    try:
        items = dbutils.fs.ls(path)
        print(f"OK - {len(items)} item(ns) encontrado(s)")
        for item in items[:5]:
            print(" ", item.path)
    except Exception as e:
        print("INDISPONÍVEL neste workspace:", str(e)[:250])

print("\n--- Unity Catalog samples")
try:
    display(dbutils.fs.ls("/Volumes/samples/databricks/datasets/"))
except Exception as e:
    print("O volume samples não está disponível ou você não tem acesso:", str(e)[:250])


--- /databricks-datasets/nyctaxi/sample/json/
OK - 88 item(ns) encontrado(s)
  dbfs:/databricks-datasets/nyctaxi/sample/json/_SUCCESS
  dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2008-12-31/
  dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2009-01-01/
  dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-11-30/
  dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-01/

--- /databricks-datasets/structured-streaming/events
OK - 50 item(ns) encontrado(s)
  dbfs:/databricks-datasets/structured-streaming/events/file-0.json
  dbfs:/databricks-datasets/structured-streaming/events/file-1.json
  dbfs:/databricks-datasets/structured-streaming/events/file-10.json
  dbfs:/databricks-datasets/structured-streaming/events/file-11.json
  dbfs:/databricks-datasets/structured-streaming/events/file-12.json

--- /databricks-datasets/retail-org/sales_orders
OK - 4 item(ns) encontrado(s)
  dbfs:/databricks-datasets/retail-or

path,name,size,modificationTime
dbfs:/Volumes/samples/databricks/datasets/COVID/,COVID/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/README.md,README.md,976,1782321760000
dbfs:/Volumes/samples/databricks/datasets/Rdatasets/,Rdatasets/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/SPARK_README.md,SPARK_README.md,3359,1782321766000
dbfs:/Volumes/samples/databricks/datasets/adult/,adult/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/airlines/,airlines/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/amazon/,amazon/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/asa/,asa/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/atlas_higgs/,atlas_higgs/,0,1790257213097
dbfs:/Volumes/samples/databricks/datasets/bikeSharing/,bikeSharing/,0,1790257213097


# 1. Dados semiestruturados e JSON

Este é o bloco mais importante para fixar a diferença entre:

```text
JSON STRING / VARIANT
        ↓
   coluna:campo

STRUCT
        ↓
   coluna.campo
```

Essa regra mental é muito útil para prova e para leitura de código, mas lembre-se de uma nuance:

```sql
raw:store.bicycle.price
```

O `:` inicia a navegação no JSON e os pontos seguintes pertencem ao **JSON path**. Isso é diferente de acessar um campo de um `STRUCT` Spark com `struct_col.campo`.

Documentação oficial:

- JSON strings e operador `:`:  
  https://docs.databricks.com/aws/en/semi-structured/json
- JSON path expression:  
  https://docs.databricks.com/aws/pt/sql/language-manual/sql-ref-json-path-expression
- Tipos complexos:  
  https://docs.databricks.com/aws/en/semi-structured/complex-types

## 1.1 NYC Taxi como texto cru

O próprio Databricks usa este dataset em exemplos de `from_json`:

```text
/databricks-datasets/nyctaxi/sample/json/
```

A ideia aqui é **não** pedir para o Spark interpretar o JSON automaticamente. Vamos ler cada linha como `STRING` para praticar as funções de parsing.

In [0]:
taxi_json_path = "/databricks-datasets/nyctaxi/sample/json/"

display(dbutils.fs.ls(taxi_json_path))

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/sample/json/_SUCCESS,_SUCCESS,0,1617328796000
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2008-12-31/,pep_pickup_date_txt=2008-12-31/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2009-01-01/,pep_pickup_date_txt=2009-01-01/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-11-30/,pep_pickup_date_txt=2019-11-30/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-01/,pep_pickup_date_txt=2019-12-01/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-02/,pep_pickup_date_txt=2019-12-02/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-03/,pep_pickup_date_txt=2019-12-03/,0,1790258358151
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-04/,pep_pickup_date_txt=2019-12-04/,0,1790258358152
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-05/,pep_pickup_date_txt=2019-12-05/,0,1790258358152
dbfs:/databricks-datasets/nyctaxi/sample/json/pep_pickup_date_txt=2019-12-06/,pep_pickup_date_txt=2019-12-06/,0,1790258358152


In [0]:
df_raw = (
    spark.read
         .format("text")
         .load(taxi_json_path)
)

display(df_raw.limit(20))

value,pep_pickup_date_txt
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:02:14"",""tpep_dropoff_datetime"":""2019-12-19 00:33:53"",""passenger_count"":1,""trip_distance"":19.32,""RatecodeID"":2,""store_and_fwd_flag"":""N"",""PULocationID"":132,""DOLocationID"":236,""payment_type"":1,""fare_amount"":52.0,""extra"":0.0,""mta_tax"":0.5,""tip_amount"":9.0,""tolls_amount"":6.12,""improvement_surcharge"":0.3,""total_amount"":70.42,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:03"",""tpep_dropoff_datetime"":""2019-12-19 00:03:20"",""passenger_count"":1,""trip_distance"":0.74,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":74,""DOLocationID"":75,""payment_type"":2,""fare_amount"":4.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":5.8,""congestion_surcharge"":0.0}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:48"",""tpep_dropoff_datetime"":""2019-12-19 00:38:42"",""passenger_count"":2,""trip_distance"":11.26,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":234,""DOLocationID"":243,""payment_type"":1,""fare_amount"":39.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":8.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":51.3,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:04"",""tpep_dropoff_datetime"":""2019-12-19 00:15:53"",""passenger_count"":2,""trip_distance"":3.54,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":48,""DOLocationID"":144,""payment_type"":1,""fare_amount"":13.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.52,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":19.32,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:27"",""tpep_dropoff_datetime"":""2019-12-19 00:11:11"",""passenger_count"":2,""trip_distance"":1.7,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":238,""DOLocationID"":236,""payment_type"":1,""fare_amount"":9.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.56,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":15.36,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:03:07"",""tpep_dropoff_datetime"":""2019-12-19 00:10:39"",""passenger_count"":1,""trip_distance"":1.79,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":231,""DOLocationID"":90,""payment_type"":1,""fare_amount"":8.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.36,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":14.16,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:36"",""tpep_dropoff_datetime"":""2019-12-19 00:52:44"",""passenger_count"":2,""trip_distance"":16.11,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":79,""DOLocationID"":201,""payment_type"":2,""fare_amount"":51.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":2.29,""improvement_surcharge"":0.3,""total_amount"":57.59,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:07"",""tpep_dropoff_datetime"":""2019-12-19 00:10:08"",""passenger_count"":2,""trip_distance"":2.07,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":140,""DOLocationID"":239,""payment_type"":2,""fare_amount"":9.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":12.8,""congestion_surcharge"":2.5}",2019-12-19
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:18"",""tpep_dropoff_datetime"":""2019-12-19 00:09:59"",""passenger_count"":2,""trip_distance"":3.3,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":141,""DOLocationID"":79,""payment_type"":2,""fare_amount"":11.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amou

In [0]:
df_raw.printSchema()

root
 |-- value: string (nullable = true)
 |-- pep_pickup_date_txt: date (nullable = true)



A coluna criada pelo reader `text` se chama normalmente `value`.

Conceitualmente:

```text
value
  ↓
STRING contendo um documento JSON
```

In [0]:
df_raw.createOrReplaceTempView("taxi_raw")
print("Temp view taxi_raw criada.")

Temp view taxi_raw criada.


## 1.2 Operador `:` Extração direta de JSON STRING

Quando a coluna contém JSON como texto, você pode usar a sintaxe:

```sql
coluna:campo
```

A documentação oficial descreve a forma geral como:

```text
<column-name>:<extraction-path>
```

In [0]:
%sql
SELECT
  value,
  value:VendorID      AS vendor_id,
  value:total_amount  AS total_amount
FROM taxi_raw
LIMIT 20

value,vendor_id,total_amount
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:02:14"",""tpep_dropoff_datetime"":""2019-12-19 00:33:53"",""passenger_count"":1,""trip_distance"":19.32,""RatecodeID"":2,""store_and_fwd_flag"":""N"",""PULocationID"":132,""DOLocationID"":236,""payment_type"":1,""fare_amount"":52.0,""extra"":0.0,""mta_tax"":0.5,""tip_amount"":9.0,""tolls_amount"":6.12,""improvement_surcharge"":0.3,""total_amount"":70.42,""congestion_surcharge"":2.5}",2,70.42
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:03"",""tpep_dropoff_datetime"":""2019-12-19 00:03:20"",""passenger_count"":1,""trip_distance"":0.74,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":74,""DOLocationID"":75,""payment_type"":2,""fare_amount"":4.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":5.8,""congestion_surcharge"":0.0}",2,5.8
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:48"",""tpep_dropoff_datetime"":""2019-12-19 00:38:42"",""passenger_count"":2,""trip_distance"":11.26,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":234,""DOLocationID"":243,""payment_type"":1,""fare_amount"":39.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":8.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":51.3,""congestion_surcharge"":2.5}",2,51.3
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:04"",""tpep_dropoff_datetime"":""2019-12-19 00:15:53"",""passenger_count"":2,""trip_distance"":3.54,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":48,""DOLocationID"":144,""payment_type"":1,""fare_amount"":13.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.52,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":19.32,""congestion_surcharge"":2.5}",2,19.32
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:27"",""tpep_dropoff_datetime"":""2019-12-19 00:11:11"",""passenger_count"":2,""trip_distance"":1.7,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":238,""DOLocationID"":236,""payment_type"":1,""fare_amount"":9.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.56,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":15.36,""congestion_surcharge"":2.5}",2,15.36
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:03:07"",""tpep_dropoff_datetime"":""2019-12-19 00:10:39"",""passenger_count"":1,""trip_distance"":1.79,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":231,""DOLocationID"":90,""payment_type"":1,""fare_amount"":8.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":2.36,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":14.16,""congestion_surcharge"":2.5}",2,14.16
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:36"",""tpep_dropoff_datetime"":""2019-12-19 00:52:44"",""passenger_count"":2,""trip_distance"":16.11,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":79,""DOLocationID"":201,""payment_type"":2,""fare_amount"":51.5,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":2.29,""improvement_surcharge"":0.3,""total_amount"":57.59,""congestion_surcharge"":2.5}",2,57.59
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:01:07"",""tpep_dropoff_datetime"":""2019-12-19 00:10:08"",""passenger_count"":2,""trip_distance"":2.07,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":140,""DOLocationID"":239,""payment_type"":2,""fare_amount"":9.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":0.0,""improvement_surcharge"":0.3,""total_amount"":12.8,""congestion_surcharge"":2.5}",2,12.8
"{""VendorID"":2,""tpep_pickup_datetime"":""2019-12-19 00:00:18"",""tpep_dropoff_datetime"":""2019-12-19 00:09:59"",""passenger_count"":2,""trip_distance"":3.3,""RatecodeID"":1,""store_and_fwd_flag"":""N"",""PULocationID"":141,""DOLocationID"":79,""payment_type"":2,""fare_amount"":11.0,""extra"":0.5,""mta_tax"":0.5,""tip_amount"":0.0,""tolls_amount"":0.0,""improvement_su

### O que observar

- `value` é uma `STRING`.
- `value:VendorID` navega no documento JSON.
- O resultado de extração de JSON STRING tende a permanecer em representação textual, então em pipelines reais você frequentemente faz `CAST` depois.

Exemplo:

```sql
value:total_amount::DOUBLE
```

In [0]:
%sql
SELECT
  value:VendorID::STRING       AS vendor_id,
  value:total_amount::DOUBLE   AS total_amount
FROM taxi_raw
LIMIT 20

vendor_id,total_amount
2,70.42
2,5.8
2,51.3
2,19.32
2,15.36
2,14.16
2,57.59
2,12.8
2,14.8
2,19.89


## 1.3 `get_json_object`

`get_json_object` também recebe uma `STRING` contendo JSON, porém usa uma expressão JSONPath iniciada em `$`.

Forma mental:

```text
JSON STRING
   +
JSONPath '$.campo'
   ↓
STRING extraída
```

Documentação oficial:

https://docs.databricks.com/aws/en/pyspark/reference/functions/get_json_object

Referência SQL:

https://docs.databricks.com/aws/en/sql/language-manual/functions/get_json_object

In [0]:
%sql
SELECT
  get_json_object(value, '$.VendorID')     AS vendor_id,
  get_json_object(value, '$.total_amount') AS total_amount
FROM taxi_raw
LIMIT 20

vendor_id,total_amount
2,70.42
2,5.8
2,51.3
2,19.32
2,15.36
2,14.16
2,57.59
2,12.8
2,14.8
2,19.89


### Comparação rápida

```text
value:VendorID
```

é navegação pelo operador de JSON do Databricks.

Enquanto:

```text
get_json_object(value, '$.VendorID')
```

usa JSONPath.

Para código novo, a documentação atual do Databricks recomenda considerar `VARIANT` + operador `:` para dados semiestruturados.

## 1.4 `from_json`: JSON STRING para STRUCT / MAP / ARRAY

`from_json` converte uma string JSON para um tipo complexo Spark.

Forma mental:

```text
JSON STRING
    │
    │ from_json
    ▼
STRUCT / MAP / ARRAY
```

Documentação:

https://docs.databricks.com/aws/en/sql/language-manual/functions/from_json

Transformação de tipos complexos:

https://docs.databricks.com/aws/en/semi-structured/complex-types

In [0]:
%sql
SELECT
  from_json(
    value,
    'VendorID STRING, total_amount DOUBLE'
  ) AS taxi
FROM taxi_raw
LIMIT 20

taxi
"List(2, 70.42)"
"List(2, 5.8)"
"List(2, 51.3)"
"List(2, 19.32)"
"List(2, 15.36)"
"List(2, 14.16)"
"List(2, 57.59)"
"List(2, 12.8)"
"List(2, 14.8)"
"List(2, 19.89)"


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW taxi_parsed AS
SELECT
  from_json(
    value,
    'VendorID STRING, total_amount DOUBLE'
  ) AS taxi
FROM taxi_raw

Agora `taxi` é um `STRUCT`.

Portanto, usamos ponto:

```sql
taxi.VendorID
taxi.total_amount
```

Aqui o `.` é acesso a campo de `STRUCT`.

In [0]:
%sql
SELECT
  taxi.VendorID     AS vendor_id,
  taxi.total_amount AS total_amount
FROM taxi_parsed
LIMIT 20

vendor_id,total_amount
2,70.42
2,5.8
2,51.3
2,19.32
2,15.36
2,14.16
2,57.59
2,12.8
2,14.8
2,19.89


### A distinção que precisa ficar automática

```text
value:VendorID
```

`value` = JSON STRING → operador `:`

```text
taxi.VendorID
```

`taxi` = STRUCT → acesso com `.`

## 1.5 JSON aninhado: por que aparece `:` e `.` na mesma expressão?

Vamos criar um exemplo pequeno e controlado.

In [0]:
nested_json = [
    ('{"store":{"bicycle":{"brand":"Trek","price":1800},"city":"Sao Paulo"}}',),
    ('{"store":{"bicycle":{"brand":"Specialized","price":2200},"city":"Campinas"}}',),
]

nested_df = spark.createDataFrame(nested_json, ["raw"])
nested_df.createOrReplaceTempView("nested_json_demo")

display(nested_df)

raw
"{""store"":{""bicycle"":{""brand"":""Trek"",""price"":1800},""city"":""Sao Paulo""}}"
"{""store"":{""bicycle"":{""brand"":""Specialized"",""price"":2200},""city"":""Campinas""}}"


In [0]:
%sql
SELECT
  raw:store.bicycle.brand         AS brand,
  raw:store.bicycle.price::DOUBLE AS price,
  raw:store.city                  AS city
FROM nested_json_demo

brand,price,city
Trek,1800.0,Sao Paulo
Specialized,2200.0,Campinas


Neste caso:

```text
raw:store.bicycle.price
   │        └───────── JSON path
   └────────────────── entra no JSON
```

Isso **não** significa que `raw` foi convertido em `STRUCT`.

Agora compare com `from_json`.

In [0]:
%sql
WITH parsed AS (
  SELECT from_json(
    raw,
    'store STRUCT<bicycle: STRUCT<brand: STRING, price: DOUBLE>, city: STRING>'
  ) AS obj
  FROM nested_json_demo
)
SELECT
  obj.store.bicycle.brand AS brand,
  obj.store.bicycle.price AS price,
  obj.store.city          AS city
FROM parsed

brand,price,city
Trek,1800.0,Sao Paulo
Specialized,2200.0,Campinas


## 1.6 `to_json`: tipo complexo para JSON STRING

É o caminho inverso de serialização:

```text
STRUCT / MAP / ARRAY / VARIANT
        │
        │ to_json
        ▼
     JSON STRING
```

Documentação oficial:

https://docs.databricks.com/aws/en/sql/language-manual/functions/to_json

In [0]:
%sql
SELECT
  taxi,
  to_json(taxi) AS taxi_json
FROM taxi_parsed
LIMIT 20

taxi,taxi_json
"List(2, 70.42)","{""VendorID"":""2"",""total_amount"":70.42}"
"List(2, 5.8)","{""VendorID"":""2"",""total_amount"":5.8}"
"List(2, 51.3)","{""VendorID"":""2"",""total_amount"":51.3}"
"List(2, 19.32)","{""VendorID"":""2"",""total_amount"":19.32}"
"List(2, 15.36)","{""VendorID"":""2"",""total_amount"":15.36}"
"List(2, 14.16)","{""VendorID"":""2"",""total_amount"":14.16}"
"List(2, 57.59)","{""VendorID"":""2"",""total_amount"":57.59}"
"List(2, 12.8)","{""VendorID"":""2"",""total_amount"":12.8}"
"List(2, 14.8)","{""VendorID"":""2"",""total_amount"":14.8}"
"List(2, 19.89)","{""VendorID"":""2"",""total_amount"":19.89}"


### Regra de memorização

```text
from_json
JSON STRING → STRUCT / MAP / ARRAY

to_json
STRUCT / MAP / ARRAY / VARIANT → JSON STRING

get_json_object
JSON STRING + JSONPath → STRING
```

## 1.7 `parse_json`: JSON STRING → VARIANT

Em runtimes modernos, `parse_json` cria um valor do tipo `VARIANT`.

```text
JSON STRING
   │
   │ parse_json
   ▼
 VARIANT
```

`VARIANT` é um tipo próprio para dados semiestruturados.

Documentação:

- `parse_json`:  
  https://docs.databricks.com/aws/en/sql/language-manual/functions/parse_json
- tipo `VARIANT`:  
  https://docs.databricks.com/aws/en/sql/language-manual/data-types/variant-type

> `parse_json` requer Databricks SQL / Databricks Runtime compatível com `VARIANT` (15.3+ na documentação atual).

In [0]:
%sql
SELECT
  parse_json(value) AS taxi_variant
FROM taxi_raw
LIMIT 20

taxi_variant
"{""DOLocationID"":236,""PULocationID"":132,""RatecodeID"":2,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0,""fare_amount"":52,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":1,""payment_type"":1,""store_and_fwd_flag"":""N"",""tip_amount"":9,""tolls_amount"":6.12,""total_amount"":70.42,""tpep_dropoff_datetime"":""2019-12-19 00:33:53"",""tpep_pickup_datetime"":""2019-12-19 00:02:14"",""trip_distance"":19.32}"
"{""DOLocationID"":75,""PULocationID"":74,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":0,""extra"":0.5,""fare_amount"":4.5,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":1,""payment_type"":2,""store_and_fwd_flag"":""N"",""tip_amount"":0,""tolls_amount"":0,""total_amount"":5.8,""tpep_dropoff_datetime"":""2019-12-19 00:03:20"",""tpep_pickup_datetime"":""2019-12-19 00:00:03"",""trip_distance"":0.74}"
"{""DOLocationID"":243,""PULocationID"":234,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":39.5,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":1,""store_and_fwd_flag"":""N"",""tip_amount"":8,""tolls_amount"":0,""total_amount"":51.3,""tpep_dropoff_datetime"":""2019-12-19 00:38:42"",""tpep_pickup_datetime"":""2019-12-19 00:00:48"",""trip_distance"":11.26}"
"{""DOLocationID"":144,""PULocationID"":48,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":13,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":1,""store_and_fwd_flag"":""N"",""tip_amount"":2.52,""tolls_amount"":0,""total_amount"":19.32,""tpep_dropoff_datetime"":""2019-12-19 00:15:53"",""tpep_pickup_datetime"":""2019-12-19 00:01:04"",""trip_distance"":3.54}"
"{""DOLocationID"":236,""PULocationID"":238,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":9,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":1,""store_and_fwd_flag"":""N"",""tip_amount"":2.56,""tolls_amount"":0,""total_amount"":15.36,""tpep_dropoff_datetime"":""2019-12-19 00:11:11"",""tpep_pickup_datetime"":""2019-12-19 00:01:27"",""trip_distance"":1.7}"
"{""DOLocationID"":90,""PULocationID"":231,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":8,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":1,""payment_type"":1,""store_and_fwd_flag"":""N"",""tip_amount"":2.36,""tolls_amount"":0,""total_amount"":14.16,""tpep_dropoff_datetime"":""2019-12-19 00:10:39"",""tpep_pickup_datetime"":""2019-12-19 00:03:07"",""trip_distance"":1.79}"
"{""DOLocationID"":201,""PULocationID"":79,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":51.5,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":2,""store_and_fwd_flag"":""N"",""tip_amount"":0,""tolls_amount"":2.29,""total_amount"":57.59,""tpep_dropoff_datetime"":""2019-12-19 00:52:44"",""tpep_pickup_datetime"":""2019-12-19 00:00:36"",""trip_distance"":16.11}"
"{""DOLocationID"":239,""PULocationID"":140,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":9,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":2,""store_and_fwd_flag"":""N"",""tip_amount"":0,""tolls_amount"":0,""total_amount"":12.8,""tpep_dropoff_datetime"":""2019-12-19 00:10:08"",""tpep_pickup_datetime"":""2019-12-19 00:01:07"",""trip_distance"":2.07}"
"{""DOLocationID"":79,""PULocationID"":141,""RatecodeID"":1,""VendorID"":2,""congestion_surcharge"":2.5,""extra"":0.5,""fare_amount"":11,""improvement_surcharge"":0.3,""mta_tax"":0.5,""passenger_count"":2,""payment_type"":2,""store_and_fwd_flag"":""N"",""tip_amount"":0,""tolls_amount"":0,""total_amount"":14.8,""tpep_dropoff_datetime"":""2019-12-19 00:09:59"",""tpep_pickup_datetime"":""2019-12-19 00:00:18"",""trip_distance"":3.3}"
"{""DOLocationID"":43,""PULocationID"":230,""Rateco

In [0]:
%sql
WITH parsed AS (
  SELECT parse_json(value) AS taxi_variant
  FROM taxi_raw
)
SELECT
  taxi_variant:VendorID::STRING      AS vendor_id,
  taxi_variant:total_amount::DOUBLE  AS total_amount
FROM parsed
LIMIT 20

vendor_id,total_amount
2,70.42
2,5.8
2,51.3
2,19.32
2,15.36
2,14.16
2,57.59
2,12.8
2,14.8
2,19.89


### `parse_json` vs `from_json`

Use a comparação abaixo como mapa mental:

| Função | Entrada | Saída | Característica |
|---|---|---|---|
| `from_json` | STRING | STRUCT/MAP/ARRAY | Você define um schema |
| `parse_json` | STRING | VARIANT | Mantém dados semiestruturados em `VARIANT` |
| `get_json_object` | STRING | STRING | Extrai um caminho JSONPath |
| `to_json` | tipo complexo / VARIANT | STRING | Serializa novamente |

Em Lakeflow Pipelines existe ainda suporte a inferência/evolução automática de schema com `from_json(..., NULL, ...)` e `schemaLocationKey`.

Documentação:

https://docs.databricks.com/aws/en/ldp/from-json-schema-evolution

## 1.8 Exercícios - Faça sem olhar os blocos anteriores

1. Extraia `VendorID` usando `:`.
2. Extraia `VendorID` usando `get_json_object`.
3. Extraia `total_amount` e converta para `DOUBLE`.
4. Use `from_json` para criar um `STRUCT`.
5. Acesse os campos do `STRUCT` com `.`.
6. Use `to_json` para serializar o `STRUCT`.
7. Use `parse_json` e extraia um campo do `VARIANT`.
8. No dataset `nested_json_demo`, extraia `store.bicycle.price` diretamente da string.
9. Converta o JSON aninhado para `STRUCT` e repita a extração.
10. Explique, em uma frase, por que `raw:store.bicycle.price` e `obj.store.bicycle.price` não representam exatamente a mesma coisa.

In [0]:
%sql


taxi
"List(2, 70.42)"
"List(2, 5.8)"
"List(2, 51.3)"
"List(2, 19.32)"
"List(2, 15.36)"
"List(2, 14.16)"
"List(2, 57.59)"
"List(2, 12.8)"
"List(2, 14.8)"
"List(2, 19.89)"
